# 08 - Outlier Detection & Treatment

**Difficulty:** ⭐⭐⭐ (Intermediate)
**Estimated Time:** 2.5 hours
**Domain:** Credit Card Fraud Detection

---

## Why Does This Matter for ML?

You're training a model to predict credit card fraud. One transaction involves a **$47,000 luxury watch purchase**. Is this:
- A data entry error (someone typed an extra zero)?
- A legitimate high-value purchase?
- A red flag for fraud?

Getting this wrong can destroy your model. Remove it blindly → your model never learns to detect high-value fraud. Keep all outliers → a single $10,000,000 data error distorts every regression coefficient.

**Outliers affect ML models in concrete ways:**
- They pull linear regression lines toward extreme values
- They inflate variance estimates, causing features to appear less informative
- They distort distance calculations in KNN, clustering, and SVM
- They force tree-based models to create many splits around extreme values

**In this notebook you will learn:**
- Four methods to detect outliers: Z-score, IQR fence, Isolation Forest, Local Outlier Factor
- When each method is appropriate
- Five treatment strategies and when to apply each
- The critical insight: in fraud detection, outliers might BE the fraud signal

---

## The Real-World Analogy: Heights in a Dataset

Imagine you're analyzing heights in a human population dataset:

- **7 feet 6 inches (229 cm):** An outlier — but perfectly real. This is NBA player Boban Marjanovic's height. Rare, but legitimate.
- **15 feet tall (457 cm):** Impossible. This is a data error — likely a unit conversion mistake or a typo.
- **4 feet 2 inches (127 cm) for an adult:** Unusual, but real — could be a person with a medical condition.

**The key distinction: outlier ≠ error.**

An outlier is simply an unusual value. It might be:
1. A genuine extreme value (real, should be kept)
2. A data collection error (wrong, should be corrected or removed)
3. The most important signal in your dataset (fraud, disease, equipment failure)

**Your job:** detect outliers, investigate their cause, and decide treatment with domain knowledge — not mechanical removal.

## Setup: Import Libraries

In [ ]:
# Core data science libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn: outlier detection models and preprocessing
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_auc_score,
    precision_score, recall_score, f1_score
)
from scipy import stats  # For Z-score computation

# Set consistent random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('All libraries loaded!')
print(f'NumPy: {np.__version__}, Pandas: {pd.__version__}')

## Step 1: Create Dataset with Natural Outliers and Artificial Errors

We'll build a transaction dataset containing:
1. **Normal transactions**: typical everyday purchases
2. **Natural legitimate outliers**: high-value transactions (car purchase, jewelry, etc.) — real, keep them
3. **Fraudulent outliers**: unusual patterns that might flag as outliers AND be fraud
4. **Artificial errors**: impossible values (negative amounts, absurd amounts) — clearly data errors

In [ ]:
def create_transaction_dataset(n_normal=2000, n_high_value=50, n_fraud=80,
                                n_errors=20, random_state=42):
    """
    Creates a dataset with four types of transactions:
    - Normal: typical everyday transactions
    - High-value: legitimate large purchases (outliers but real)
    - Fraud: fraudulent transactions with unusual patterns
    - Errors: impossible/erroneous values that should be flagged
    """
    rng = np.random.RandomState(random_state)

    # --- NORMAL TRANSACTIONS ---
    # Log-normal amounts: most transactions between $5 and $500
    normal_amount = rng.lognormal(mean=3.8, sigma=1.0, size=n_normal)
    # Number of transactions in last 30 days per customer
    normal_num_tx = rng.poisson(lam=15, size=n_normal).clip(1, 60)
    # Average transaction amount for this customer (historical)
    normal_avg_amt = rng.lognormal(mean=3.5, sigma=0.8, size=n_normal)
    # Merchant risk score (0=safe, 1=risky)
    normal_risk = rng.beta(a=2, b=6, size=n_normal)  # Mostly low risk
    # Transaction hour
    normal_hour = rng.normal(14, 4, n_normal).clip(6, 23)
    normal_label = [0] * n_normal  # Not fraud
    normal_type = ['normal'] * n_normal

    # --- HIGH-VALUE LEGITIMATE TRANSACTIONS (natural outliers) ---
    # Expensive but real: car purchase ($15,000-$80,000), jewelry, medical bills
    hv_amount = rng.uniform(15000, 90000, size=n_high_value)
    hv_num_tx = rng.poisson(lam=12, size=n_high_value).clip(1, 40)
    hv_avg_amt = rng.lognormal(mean=4.5, sigma=0.9, size=n_high_value)  # Higher avg = wealthy customer
    hv_risk = rng.beta(a=1, b=5, size=n_high_value)  # Low risk merchant (car dealership, etc.)
    hv_hour = rng.normal(13, 3, n_high_value).clip(8, 20)  # During business hours
    hv_label = [0] * n_high_value  # NOT fraud — legitimate
    hv_type = ['high_value_legit'] * n_high_value

    # --- FRAUDULENT TRANSACTIONS ---
    # Two patterns: small test charges AND large fraudulent purchases
    fraud_amount_test  = rng.uniform(0.5, 5, size=n_fraud // 2)   # "Card testing" — tiny charges
    fraud_amount_large = rng.uniform(1000, 8000, size=n_fraud // 2)  # Large fraudulent purchase
    fraud_amount = np.concatenate([fraud_amount_test, fraud_amount_large])
    fraud_num_tx = rng.poisson(lam=25, size=n_fraud).clip(5, 80)  # High transaction velocity
    fraud_avg_amt = rng.lognormal(mean=2.8, sigma=0.7, size=n_fraud)  # Low avg (new card abuse)
    fraud_risk = rng.beta(a=5, b=2, size=n_fraud)  # High-risk merchants
    fraud_hour = rng.normal(3, 2.5, n_fraud).clip(0, 8)  # Late night / early morning
    fraud_label = [1] * n_fraud
    fraud_type = ['fraud'] * n_fraud

    # --- DATA ERRORS (impossible values) ---
    # These should be caught and removed/corrected before modeling
    error_amount = np.array(
        [-50, -12, -3, -100, -0.01,        # Negative amounts (impossible)
          1e7, 2.5e6, 5e6, 3e7, 8e6,        # Absurdly high (likely unit errors: cents not dollars)
          0, 0, 0, 0, 0,                     # Zero-dollar transactions (often errors)
          999999, 1234567, 7654321, 9999999, 4444444]  # Suspicious round-number entries
    )
    error_num_tx = rng.poisson(lam=10, size=n_errors).clip(1, 30)
    error_avg_amt = rng.lognormal(mean=3.0, sigma=1.0, size=n_errors)
    error_risk = rng.beta(a=2, b=4, size=n_errors)
    error_hour = rng.uniform(0, 23, size=n_errors)
    error_label = [0] * n_errors  # Unknown — classified as not-fraud but values are wrong
    error_type = ['data_error'] * n_errors

    # Combine all transaction types into one DataFrame
    df = pd.DataFrame({
        'amount': np.concatenate([
            normal_amount, hv_amount, fraud_amount, error_amount
        ]),
        'num_transactions_30d': np.concatenate([
            normal_num_tx, hv_num_tx, fraud_num_tx, error_num_tx
        ]),
        'avg_amount_historical': np.concatenate([
            normal_avg_amt, hv_avg_amt, fraud_avg_amt, error_avg_amt
        ]),
        'merchant_risk': np.concatenate([
            normal_risk, hv_risk, fraud_risk, error_risk
        ]),
        'hour': np.concatenate([
            normal_hour, hv_hour, fraud_hour, error_hour
        ]),
        'is_fraud': np.concatenate([normal_label, hv_label, fraud_label, error_label]),
        'transaction_type': normal_type + hv_type + fraud_type + error_type
    })

    # Shuffle rows
    df = df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    return df

df = create_transaction_dataset()

print(f'Dataset shape: {df.shape}')
print(f'\nTransaction type counts:')
print(df['transaction_type'].value_counts())
print(f'\nAmount statistics:')
print(df['amount'].describe().round(2))
print(f'\nMin amount: ${df["amount"].min():.2f}  Max: ${df["amount"].max():,.0f}')

## Step 2: Visualize the Raw Data Distribution

Before detecting outliers, always visualize the data. The outlier pattern becomes immediately obvious.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# --- Plot 1: Raw amount distribution (full scale) ---
axes[0, 0].hist(df['amount'], bins=100, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0, 0].set_title('Transaction Amounts — Raw Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Amount ($)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_xlim(-500, df['amount'].max() + 100000)

# Add vertical lines showing where "normal" transactions end
axes[0, 0].axvline(x=1000, color='orange', linestyle='--', linewidth=1.5, label='$1,000 threshold')
axes[0, 0].axvline(x=0, color='red', linestyle='--', linewidth=1.5, label='$0 line (errors below)')
axes[0, 0].legend()

# --- Plot 2: Log-scale distribution (see structure better) ---
# Many outlier patterns are hidden when the full scale includes extreme values
# Log transform reveals the true distribution shape
positive_amounts = df[df['amount'] > 0]['amount']  # Log requires positive values
axes[0, 1].hist(np.log1p(positive_amounts), bins=80, color='steelblue',
                edgecolor='white', linewidth=0.3)
axes[0, 1].set_title('Transaction Amounts — Log Scale (log(1+amount))',
                     fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('log(1 + Amount)')
axes[0, 1].set_ylabel('Count')

# Mark the high-value transaction region
axes[0, 1].axvline(x=np.log1p(1000), color='orange', linestyle='--', linewidth=1.5,
                   label=f'log($1,000) = {np.log1p(1000):.1f}')
axes[0, 1].axvline(x=np.log1p(15000), color='red', linestyle='--', linewidth=1.5,
                   label=f'log($15,000) = {np.log1p(15000):.1f}')
axes[0, 1].legend()

# --- Plot 3: Scatter plot — amount vs num_transactions ---
# Color-code by transaction type so we can see where outliers live
type_colors = {'normal': '#2ecc71', 'high_value_legit': '#3498db',
               'fraud': '#e74c3c', 'data_error': '#f39c12'}

for tx_type, color in type_colors.items():
    mask = df['transaction_type'] == tx_type
    axes[1, 0].scatter(
        df.loc[mask, 'amount'],
        df.loc[mask, 'num_transactions_30d'],
        alpha=0.5, c=color, s=20, label=tx_type, edgecolors='none'
    )

axes[1, 0].set_title('Amount vs Transaction Velocity', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Transaction Amount ($)')
axes[1, 0].set_ylabel('Num Transactions in Last 30 Days')
axes[1, 0].set_xscale('symlog')  # Symmetric log scale handles negative values
axes[1, 0].legend()

# --- Plot 4: Box plot by transaction type ---
# Box plots are excellent for identifying outliers visually (dots beyond the whiskers)
type_amounts = [df[df['transaction_type'] == t]['amount'].clip(-200, 5000)
                for t in ['normal', 'high_value_legit', 'fraud', 'data_error']]
bp = axes[1, 1].boxplot(type_amounts,
                         labels=['Normal', 'High-Value\nLegit', 'Fraud', 'Data\nErrors'],
                         patch_artist=True, notch=False)

# Color each box
box_colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

axes[1, 1].set_title('Amount Distribution by Transaction Type\n(capped at $5,000 for visibility)',
                      fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Transaction Amount ($)')
axes[1, 1].axhline(y=0, color='black', linestyle='--', alpha=0.5, linewidth=1)

plt.suptitle('Credit Card Transactions — Understanding the Data Distribution',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Key observations:')
print(f'  - Min amount: ${df["amount"].min():.2f} (negative = data error)')
print(f'  - Max amount: ${df["amount"].max():,.0f} (likely data error - absurdly high)')
print(f'  - Transactions with negative amounts: {(df["amount"] < 0).sum()}')
print(f'  - Transactions > $100,000: {(df["amount"] > 100000).sum()}')

## Method 1: Z-Score Detection

### Plain English

**Z-score** measures "how many standard deviations is this value from the mean?"

Formula: `z = (value - mean) / standard_deviation`

- z = 0: exactly average
- z = 2: 2 standard deviations above mean (top ~2.3% in normal distribution)
- z = 3: 3 standard deviations above mean (top ~0.13%) → we call this an outlier

**Limitation:** Z-score assumes the data is **normally distributed** (bell curve). Transaction amounts are right-skewed (many small, few very large). On skewed data, Z-score performs poorly.

In [ ]:
# ============================================================
# METHOD 1: Z-Score Outlier Detection
# ============================================================

# Compute z-scores for the 'amount' feature
# scipy.stats.zscore computes (x - mean) / std for each element
z_scores_amount = np.abs(stats.zscore(df['amount']))

# Standard threshold: |z| > 3 is considered an outlier
Z_THRESHOLD = 3
z_outlier_mask = z_scores_amount > Z_THRESHOLD

print(f'Z-Score Detection (threshold = {Z_THRESHOLD})')
print(f'  Total transactions:  {len(df)}')
print(f'  Outliers detected:   {z_outlier_mask.sum()} ({z_outlier_mask.mean()*100:.2f}%)')
print()

# See what was flagged — check if we're catching data errors and missing fraud
outlier_breakdown = df[z_outlier_mask]['transaction_type'].value_counts()
print('Breakdown of flagged outliers by type:')
print(outlier_breakdown)
print()

# Show some flagged transactions
print('Sample flagged transactions (sorted by z-score):')
df_z = df.copy()
df_z['z_score'] = z_scores_amount
print(df_z[z_outlier_mask][['amount', 'transaction_type', 'is_fraud', 'z_score']]
      .sort_values('z_score', ascending=False)
      .head(15).round(2).to_string())

print()
# What did Z-score MISS? (false negatives — outliers not flagged)
# High-value legitimate transactions might NOT have |z| > 3 if they're not extreme enough
print('High-value legitimate transactions NOT flagged as outliers (missed):')
hv_missed = df[(df['transaction_type'] == 'high_value_legit') & (~z_outlier_mask)]
print(f'  {len(hv_missed)} high-value legit transactions below z-threshold')
print(f'  Their amount range: ${hv_missed["amount"].min():,.0f} – ${hv_missed["amount"].max():,.0f}')

print()
print('NOTE: Z-score struggles with skewed distributions because:')
print('  - The mean is pulled toward extreme values')
print('  - The std is inflated by those same extreme values')
print('  - Result: even extreme outliers may not exceed 3σ on heavily skewed data')

In [ ]:
# Visualize Z-score detection results
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Left: Histogram with outlier regions shaded ---
# Use only amounts < 20000 for visibility (extreme error values distort the plot)
amount_viz = df[df['amount'].between(0, 20000)]['amount']
z_viz_mask = z_outlier_mask[df['amount'].between(0, 20000)]

normal_amounts = amount_viz[~z_viz_mask]
outlier_amounts = amount_viz[z_viz_mask]

axes[0].hist(normal_amounts, bins=60, color='#3498db', alpha=0.7,
             label='Normal (|z| ≤ 3)')
axes[0].hist(outlier_amounts, bins=30, color='#e74c3c', alpha=0.8,
             label=f'Outlier (|z| > 3): {z_outlier_mask.sum()} pts')

# Mark Z-score threshold on the x-axis
z_mean = df['amount'].mean()
z_std = df['amount'].std()
upper_z3 = z_mean + 3 * z_std
lower_z3 = z_mean - 3 * z_std

axes[0].axvline(x=upper_z3, color='red', linestyle='--', linewidth=1.5,
               label=f'μ + 3σ = ${upper_z3:,.0f}')
if lower_z3 > 0:
    axes[0].axvline(x=lower_z3, color='red', linestyle=':', linewidth=1.5,
                   label=f'μ - 3σ = ${lower_z3:,.0f}')

axes[0].set_title('Z-Score Detection on Transaction Amount\n(capped at $20,000 for visibility)',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Transaction Amount ($)')
axes[0].set_ylabel('Count')
axes[0].legend(fontsize=9)

# --- Right: Z-score vs actual values scatter ---
# Show how z-scores correlate with raw values — helps see the threshold effect
df_plot = df[df['amount'].between(-500, 50000)].copy()
z_scores_plot = np.abs(stats.zscore(df['amount']))[df['amount'].between(-500, 50000)]

colors_map = df_plot['transaction_type'].map({
    'normal': '#2ecc71', 'high_value_legit': '#3498db',
    'fraud': '#e74c3c', 'data_error': '#f39c12'
})

axes[1].scatter(df_plot['amount'], z_scores_plot,
               c=colors_map, alpha=0.5, s=20)
axes[1].axhline(y=Z_THRESHOLD, color='red', linestyle='--', linewidth=2,
               label=f'Z-score threshold = {Z_THRESHOLD}')
axes[1].set_xlabel('Transaction Amount ($)')
axes[1].set_ylabel('|Z-Score|')
axes[1].set_title('Z-Score vs Amount (colored by type)', fontsize=12, fontweight='bold')

# Add legend manually
legend_patches = [
    mpatches.Patch(color='#2ecc71', label='Normal'),
    mpatches.Patch(color='#3498db', label='High-Value Legit'),
    mpatches.Patch(color='#e74c3c', label='Fraud'),
    mpatches.Patch(color='#f39c12', label='Data Error'),
]
axes[1].legend(handles=legend_patches, fontsize=9)
axes[1].legend([plt.Line2D([0],[0],color='red',linestyle='--',linewidth=2)] + legend_patches,
               [f'Threshold (z={Z_THRESHOLD})'] + ['Normal','High-Value Legit','Fraud','Data Error'],
               fontsize=9)

plt.tight_layout()
plt.show()

## Method 2: IQR Fence (Tukey's Method)

### Plain English

**IQR** = Interquartile Range = Q3 − Q1 = the range containing the **middle 50%** of your data.

Tukey's fences define outlier boundaries:
- **Lower fence:** Q1 − 1.5 × IQR
- **Upper fence:** Q3 + 1.5 × IQR

Anything beyond these fences is a potential outlier.

**Why IQR is better than Z-score for skewed data:**
- IQR is based on medians/percentiles, not means/std
- Extreme values don't distort the calculation
- Works on any distribution shape — no normality assumption

**Analogy:** Instead of measuring "how far from average height?" (Z-score), measure "does this height fall outside the range of the middle 50% of humans?" (IQR).

In [ ]:
# ============================================================
# METHOD 2: IQR Fence (Tukey's Method)
# ============================================================

def iqr_outliers(series, multiplier=1.5):
    """
    Returns boolean mask: True where value is an IQR outlier.
    multiplier=1.5 is standard; use 3.0 for 'extreme outliers' only.
    """
    Q1 = series.quantile(0.25)  # 25th percentile
    Q3 = series.quantile(0.75)  # 75th percentile
    IQR = Q3 - Q1               # Interquartile range

    lower_fence = Q1 - multiplier * IQR  # Below this: potential outlier
    upper_fence = Q3 + multiplier * IQR  # Above this: potential outlier

    # Flag values outside the fences
    outlier_mask = (series < lower_fence) | (series > upper_fence)

    return outlier_mask, lower_fence, upper_fence, Q1, Q3, IQR

# Apply IQR detection to 'amount'
iqr_mask, lower, upper, Q1, Q3, IQR = iqr_outliers(df['amount'], multiplier=1.5)

print(f'IQR Detection (multiplier = 1.5)')
print(f'  Q1 = ${Q1:.2f}')
print(f'  Q3 = ${Q3:.2f}')
print(f'  IQR = ${IQR:.2f}')
print(f'  Lower fence: ${lower:.2f}')
print(f'  Upper fence: ${upper:.2f}')
print()
print(f'  Outliers detected: {iqr_mask.sum()} ({iqr_mask.mean()*100:.2f}%)')
print()

# Breakdown by transaction type
print('IQR-flagged outliers by transaction type:')
print(df[iqr_mask]['transaction_type'].value_counts())
print()

# Compare Z-score vs IQR on the same data
print('COMPARISON: Z-score vs IQR Detection')
print(f'  Z-score (|z| > 3) flagged:  {z_outlier_mask.sum()} transactions')
print(f'  IQR (1.5x multiplier) flagged: {iqr_mask.sum()} transactions')

# Transactions flagged by BOTH methods
both = z_outlier_mask & iqr_mask
only_z = z_outlier_mask & ~iqr_mask
only_iqr = iqr_mask & ~z_outlier_mask

print(f'  Flagged by BOTH:       {both.sum()}')
print(f'  Only Z-score flagged:  {only_z.sum()}')
print(f'  Only IQR flagged:      {only_iqr.sum()}')
print()
print('Note: IQR catches more outliers because the amount distribution is right-skewed.')
print('The IQR method is not misled by the extreme values distorting the mean/std.')

In [ ]:
# Visualize IQR detection with box plot and histogram
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Left: Box plot with annotations ---
# Use a capped version for visibility
amount_capped = df['amount'].clip(-1000, 8000)

bp = axes[0].boxplot(amount_capped, vert=True, patch_artist=True,
                     flierprops=dict(marker='o', markerfacecolor='#e74c3c',
                                     markersize=4, linestyle='none', alpha=0.5))
bp['boxes'][0].set_facecolor('#3498db')
bp['boxes'][0].set_alpha(0.6)

# Annotate key statistics
annotations = {
    f'Q1 = ${Q1:.0f}': (1.05, Q1),
    f'Median = ${df["amount"].median():.0f}': (1.05, df['amount'].median()),
    f'Q3 = ${Q3:.0f}': (1.05, Q3),
    f'Upper fence = ${upper:.0f}': (1.05, min(upper, 8000)),
}
for text, (x_pos, y_pos) in annotations.items():
    axes[0].annotate(text, xy=(x_pos, y_pos),
                    fontsize=9, va='center', color='navy')

axes[0].axhline(y=upper, color='red', linestyle='--', linewidth=1.5, alpha=0.7,
               label=f'Upper fence (Q3 + 1.5×IQR) = ${upper:.0f}')
axes[0].axhline(y=lower, color='red', linestyle=':', linewidth=1.5, alpha=0.7,
               label=f'Lower fence = ${lower:.2f}')
axes[0].set_title('Box Plot — IQR Fence Detection\n(capped at -$1,000 to $8,000)',
                  fontsize=12, fontweight='bold')
axes[0].set_ylabel('Transaction Amount ($)')
axes[0].set_xticklabels(['Transaction Amount'])
axes[0].legend(fontsize=8, loc='upper right')

# --- Right: Side-by-side comparison Z-score vs IQR ---
# Venn-style bar chart: how many does each method catch?
methods = ['Z-score\n(|z|>3)', 'IQR\n(1.5x)', 'Both\nMethods']
counts = [z_outlier_mask.sum(), iqr_mask.sum(), both.sum()]
colors = ['#3498db', '#e74c3c', '#9b59b6']

bars = axes[1].bar(methods, counts, color=colors, alpha=0.8,
                   edgecolor='black', linewidth=0.8)
for bar, count in zip(bars, counts):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(count), ha='center', va='bottom', fontweight='bold', fontsize=12)

axes[1].set_title('Z-Score vs IQR: Outliers Detected', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Number of Outliers Detected')
axes[1].set_ylim(0, max(counts) + 20)

# Add breakdown by transaction type for each method
z_err = (z_outlier_mask & (df['transaction_type'] == 'data_error')).sum()
iqr_err = (iqr_mask & (df['transaction_type'] == 'data_error')).sum()
axes[1].text(0, counts[0]/2, f'{z_err} errors', ha='center', color='white',
            fontweight='bold', fontsize=9)
axes[1].text(1, counts[1]/2, f'{iqr_err} errors', ha='center', color='white',
            fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

## Method 3: Isolation Forest

### Plain English

**Isolation Forest** is a machine learning algorithm based on a clever insight:

> **Outliers are easier to isolate than normal points.**

Here's how it works:
1. Randomly pick a feature
2. Randomly pick a split point between the feature's min and max
3. Divide the data into two groups
4. Repeat (it's building random decision trees)
5. Count how many splits it took to isolate each point

**Outliers need fewer splits** — they're already isolated from the crowd. Normal points are densely packed and need many splits to isolate.

**Analogy:** Finding a fraud suspect in a crowd. A normal person looks like everyone else and takes many questions to identify. A fraudster has many unusual characteristics — you identify them quickly.

**Advantage:** No distribution assumptions, works well in high dimensions, handles multivariate outliers.

In [ ]:
# ============================================================
# METHOD 3: Isolation Forest
# ============================================================

# Use two features for visualization: amount and num_transactions_30d
# In practice, you'd use all features
features_for_if = ['amount', 'num_transactions_30d', 'merchant_risk', 'hour']
X_if = df[features_for_if].copy()

# Handle data errors before fitting: Isolation Forest can't handle NaN,
# and extreme error values will distort the contamination parameter
# We'll clip extreme errors to a reasonable range for this analysis
X_if_clipped = X_if.copy()
X_if_clipped['amount'] = X_if_clipped['amount'].clip(-1000, 200000)

# contamination: expected proportion of outliers in the dataset
# We know we injected ~70/2150 ≈ 3.3% outliers (errors + high_value + fraud)
# Setting it slightly higher than true proportion is typical practice
CONTAMINATION = 0.05  # Expect ~5% outliers

iso_forest = IsolationForest(
    contamination=CONTAMINATION,  # Expected fraction of outliers
    n_estimators=100,              # Number of trees in the ensemble
    max_samples='auto',            # Samples per tree (default: min(256, n_samples))
    random_state=RANDOM_STATE
)

# Fit and predict: returns +1 (inlier) or -1 (outlier)
if_predictions = iso_forest.fit_predict(X_if_clipped)

# Convert: -1 → True (outlier), +1 → False (inlier)
if_outlier_mask = if_predictions == -1

# Anomaly scores: more negative = more anomalous
# decision_function returns the anomaly score before thresholding
if_scores = iso_forest.decision_function(X_if_clipped)

print(f'Isolation Forest Results (contamination={CONTAMINATION})')
print(f'  Outliers detected: {if_outlier_mask.sum()} ({if_outlier_mask.mean()*100:.2f}%)')
print()
print('Breakdown by transaction type:')
print(df[if_outlier_mask]['transaction_type'].value_counts())
print()

# How many FRAUD transactions did Isolation Forest flag?
fraud_caught = (if_outlier_mask & (df['is_fraud'] == 1)).sum()
total_fraud = (df['is_fraud'] == 1).sum()
print(f'Fraud detection: {fraud_caught}/{total_fraud} fraud cases flagged')
print(f'  ({fraud_caught/total_fraud*100:.1f}% of all frauds detected as anomalies)')

# How many legitimate transactions were flagged (false positives)?
fp = (if_outlier_mask & (df['is_fraud'] == 0)).sum()
total_legit = (df['is_fraud'] == 0).sum()
print(f'False positives: {fp}/{total_legit} legitimate transactions flagged')

In [ ]:
# Visualize Isolation Forest results in 2D (amount vs num_transactions)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left: Anomaly score heatmap-style scatter ---
# Use amount and num_transactions as the 2D view
# Color = anomaly score (more negative = more anomalous)
plot_mask = df['amount'].between(-500, 100000)  # Exclude extreme errors for visibility

sc = axes[0].scatter(
    np.log1p(np.abs(df.loc[plot_mask, 'amount'])) * np.sign(df.loc[plot_mask, 'amount']),
    df.loc[plot_mask, 'num_transactions_30d'],
    c=if_scores[plot_mask],   # Color by anomaly score
    cmap='RdYlGn',             # Red = anomalous, Green = normal
    alpha=0.6, s=25,
    vmin=if_scores.min(), vmax=if_scores.max()
)

plt.colorbar(sc, ax=axes[0], label='Anomaly Score (lower = more anomalous)')
axes[0].set_xlabel('log(|Amount|) × sign (sign-log scale)')
axes[0].set_ylabel('Num Transactions in Last 30 Days')
axes[0].set_title('Isolation Forest — Anomaly Scores\n(Red = anomalous, Green = normal)',
                  fontsize=12, fontweight='bold')

# --- Right: Outlier classification scatter ---
type_colors_if = {
    'normal': '#2ecc71',
    'high_value_legit': '#3498db',
    'fraud': '#e74c3c',
    'data_error': '#f39c12'
}

for tx_type, color in type_colors_if.items():
    mask = (df['transaction_type'] == tx_type) & plot_mask
    inlier_mask = mask & ~if_outlier_mask
    outlier_mask_type = mask & if_outlier_mask

    # Inliers: regular marker
    if inlier_mask.sum() > 0:
        axes[1].scatter(
            np.log1p(np.abs(df.loc[inlier_mask, 'amount'])) * np.sign(df.loc[inlier_mask, 'amount']),
            df.loc[inlier_mask, 'num_transactions_30d'],
            c=color, alpha=0.3, s=15, marker='o', label=f'{tx_type} (inlier)'
        )
    # Outliers: star marker, larger
    if outlier_mask_type.sum() > 0:
        axes[1].scatter(
            np.log1p(np.abs(df.loc[outlier_mask_type, 'amount'])) * np.sign(df.loc[outlier_mask_type, 'amount']),
            df.loc[outlier_mask_type, 'num_transactions_30d'],
            c=color, alpha=0.9, s=80, marker='*',
            label=f'{tx_type} (OUTLIER: {outlier_mask_type.sum()})'
        )

axes[1].set_xlabel('log(|Amount|) × sign')
axes[1].set_ylabel('Num Transactions in Last 30 Days')
axes[1].set_title('Isolation Forest — Flagged Outliers (★) by Type',
                  fontsize=12, fontweight='bold')
axes[1].legend(fontsize=7, loc='upper left', ncol=2)

plt.tight_layout()
plt.show()

## Method 4: Local Outlier Factor (LOF)

### Plain English

**LOF** asks a different question: "Is this point in a sparse neighborhood compared to its neighbors?"

Step by step:
1. Find the k nearest neighbors of a point
2. Measure the density of those neighbors (how tightly packed they are)
3. Compare the density at the point vs. the density of its neighbors
4. If the point is in a sparse area while its neighbors are dense → it's an outlier

**Why LOF catches things Isolation Forest might miss:**
LOF can detect **local outliers** — points that are outliers within their neighborhood cluster, but might look normal globally.

**Analogy:** A $1,000 purchase is not unusual globally. But if that person normally spends $20 and suddenly makes a $1,000 purchase, it's locally anomalous — an outlier compared to their own neighborhood of similar customers.

In [ ]:
# ============================================================
# METHOD 4: Local Outlier Factor (LOF)
# ============================================================

# Scale features: LOF uses distance metrics, so scale matters
# We use the same features as Isolation Forest for comparison
scaler_lof = StandardScaler()
X_lof = scaler_lof.fit_transform(X_if_clipped)

# n_neighbors: number of nearest neighbors to consider for density estimation
# Larger n_neighbors → smoother density estimate, less sensitive to local noise
# contamination: same meaning as Isolation Forest — expected fraction of outliers
lof = LocalOutlierFactor(
    n_neighbors=20,             # Consider 20 nearest neighbors
    contamination=CONTAMINATION,  # Same 5% threshold as IF for fair comparison
    novelty=False               # False means we're doing outlier detection on training data
)

# LOF uses fit_predict directly (novelty=False)
# Returns +1 (inlier) or -1 (outlier)
lof_predictions = lof.fit_predict(X_lof)
lof_outlier_mask = lof_predictions == -1

# LOF scores: negative_outlier_factor_ (more negative = more anomalous)
lof_scores = -lof.negative_outlier_factor_  # Negate so higher = more anomalous

print(f'Local Outlier Factor Results (n_neighbors=20, contamination={CONTAMINATION})')
print(f'  Outliers detected: {lof_outlier_mask.sum()} ({lof_outlier_mask.mean()*100:.2f}%)')
print()
print('Breakdown by transaction type:')
print(df[lof_outlier_mask]['transaction_type'].value_counts())
print()

# --- Agreement between Isolation Forest and LOF ---
both_flag = if_outlier_mask & lof_outlier_mask
only_if = if_outlier_mask & ~lof_outlier_mask
only_lof = lof_outlier_mask & ~if_outlier_mask
neither = ~if_outlier_mask & ~lof_outlier_mask

print('AGREEMENT BETWEEN METHODS:')
print(f'  Both flag as outlier:       {both_flag.sum()}')
print(f'  Only Isolation Forest:      {only_if.sum()}')
print(f'  Only LOF:                   {only_lof.sum()}')
print(f'  Neither flags:              {neither.sum()}')
print()
print('FRAUD DETECTION BY EACH METHOD:')
print(f'  Isolation Forest caught:  {(if_outlier_mask & (df["is_fraud"]==1)).sum()}/{(df["is_fraud"]==1).sum()} frauds')
print(f'  LOF caught:               {(lof_outlier_mask & (df["is_fraud"]==1)).sum()}/{(df["is_fraud"]==1).sum()} frauds')
print(f'  Either method caught:     {((if_outlier_mask | lof_outlier_mask) & (df["is_fraud"]==1)).sum()}/{(df["is_fraud"]==1).sum()} frauds')

In [ ]:
# Visualize agreement between Isolation Forest and LOF
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left: Agreement scatter plot ---
agreement_colors = {
    'Both': '#9b59b6',       # Purple: both methods agree it's an outlier
    'IF only': '#e67e22',    # Orange: only Isolation Forest flags it
    'LOF only': '#e74c3c',   # Red: only LOF flags it
    'Neither': '#95a5a6',    # Gray: neither method flags it
}

# Create an agreement category for each transaction
agreement_cat = pd.Series('Neither', index=df.index)
agreement_cat[only_if] = 'IF only'
agreement_cat[only_lof] = 'LOF only'
agreement_cat[both_flag] = 'Both'

# Use log-scale amount vs merchant_risk for the 2D view
plot_mask2 = df['amount'].between(0.01, 100000)

for cat, color in agreement_colors.items():
    m = (agreement_cat == cat) & plot_mask2
    size = 60 if cat != 'Neither' else 15
    alpha = 0.9 if cat != 'Neither' else 0.2
    axes[0].scatter(
        np.log1p(df.loc[m, 'amount']),
        df.loc[m, 'merchant_risk'],
        c=color, s=size, alpha=alpha, label=f'{cat} ({m.sum()})',
        zorder=3 if cat != 'Neither' else 1
    )

axes[0].set_xlabel('log(1 + Amount)')
axes[0].set_ylabel('Merchant Risk Score')
axes[0].set_title('Isolation Forest vs LOF Agreement\n(Colored by consensus)',
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)

# --- Right: Venn-style bar chart of method agreement ---
categories = ['Both\nMethods', 'IF Only', 'LOF Only']
counts_agreement = [both_flag.sum(), only_if.sum(), only_lof.sum()]

# Also show how many fraud cases in each category
fraud_in_both = (both_flag & (df['is_fraud']==1)).sum()
fraud_in_if_only = (only_if & (df['is_fraud']==1)).sum()
fraud_in_lof_only = (only_lof & (df['is_fraud']==1)).sum()

x = np.arange(3)
width = 0.35
b1 = axes[1].bar(x - width/2, counts_agreement, width,
                  color=['#9b59b6', '#e67e22', '#e74c3c'],
                  label='All Flagged', alpha=0.8, edgecolor='black')
b2 = axes[1].bar(x + width/2, [fraud_in_both, fraud_in_if_only, fraud_in_lof_only], width,
                  color=['#9b59b6', '#e67e22', '#e74c3c'],
                  label='Fraud in Flagged', alpha=0.4, edgecolor='black', hatch='//')

# Add count labels
for bar in b1:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                str(int(bar.get_height())), ha='center', fontsize=10, fontweight='bold')
for bar in b2:
    if bar.get_height() > 0:
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                    str(int(bar.get_height())), ha='center', fontsize=10, color='darkred')

axes[1].set_title('Method Agreement: Total vs Fraud Caught',
                  fontsize=12, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(categories)
axes[1].set_ylabel('Number of Transactions')
axes[1].legend()

plt.tight_layout()
plt.show()

## Treatment Strategies

Detection finds outliers. Treatment decides what to do with them. There is no universally correct treatment — **domain knowledge matters more than any formula**.

| Strategy | When to use | Risk |
|----------|------------|------|
| **Remove** | Clear data errors (negative amounts, impossible values) | Loss of information |
| **Cap/Winsorize** | Genuine outliers that inflate variance | Distorts true extreme values |
| **Log Transform** | Right-skewed distributions (transaction amounts) | Changes feature meaning |
| **Keep as-is** | Outliers are the signal (fraud!) | Model may struggle with extremes |
| **Binary indicator** | Preserve signal + reduce distortion | Adds dimensionality |

### The Fraud Detection Dilemma

In fraud detection, high-value unusual transactions are often **the fraud itself**. Removing or capping them would destroy the most important signal. The right approach is often to **create an indicator feature** that preserves the signal while reducing numerical distortion.

In [ ]:
# ============================================================
# TREATMENT STRATEGY 1: Remove clear data errors
# ============================================================

print('TREATMENT 1: Remove clear data errors')
print('-' * 50)

# Rule 1: Remove negative amounts (impossible for a purchase)
negative_mask = df['amount'] < 0
print(f'  Negative amounts:      {negative_mask.sum()} rows → REMOVE')

# Rule 2: Remove zero-amount transactions (usually errors)
zero_mask = df['amount'] == 0
print(f'  Zero-amount:           {zero_mask.sum()} rows → REMOVE')

# Rule 3: Remove clearly impossible amounts (> $5M — even car collections don't cost this)
impossible_mask = df['amount'] > 5_000_000
print(f'  Amounts > $5M:         {impossible_mask.sum()} rows → REMOVE')

# Combine all error masks
error_mask = negative_mask | zero_mask | impossible_mask
df_clean = df[~error_mask].copy()

print(f'\n  Rows before removal: {len(df)}')
print(f'  Rows removed:        {error_mask.sum()}')
print(f'  Rows after:          {len(df_clean)}')
print()

# ============================================================
# TREATMENT STRATEGY 2: Capping / Winsorizing
# ============================================================

print('TREATMENT 2: Capping / Winsorizing')
print('-' * 50)

# Winsorizing: cap values at a percentile threshold
# Values beyond the 99th percentile get replaced with the 99th percentile value
# This reduces the influence of extreme values without removing rows
p1 = df_clean['amount'].quantile(0.01)   # 1st percentile (lower cap)
p99 = df_clean['amount'].quantile(0.99)  # 99th percentile (upper cap)

print(f'  1st percentile: ${p1:.2f}')
print(f'  99th percentile: ${p99:,.2f}')

# np.clip replaces values below lower with lower, above upper with upper
df_clean['amount_winsorized'] = df_clean['amount'].clip(lower=p1, upper=p99)

# Count how many rows were actually changed
changed = ((df_clean['amount'] < p1) | (df_clean['amount'] > p99)).sum()
print(f'  Rows affected by capping: {changed}')
print(f'  Max before: ${df_clean["amount"].max():,.0f}  →  Max after: ${df_clean["amount_winsorized"].max():,.0f}')
print()

# ============================================================
# TREATMENT STRATEGY 3: Log Transform
# ============================================================

print('TREATMENT 3: Log Transform')
print('-' * 50)

# log1p = log(1 + x) — the +1 handles x=0 gracefully (log(0) is undefined)
# This compresses the scale dramatically:
#   $1     → log(2)    ≈ 0.69
#   $100   → log(101)  ≈ 4.62
#   $10000 → log(10001)≈ 9.21
# Now $100 and $10,000 are separated by only 4.6 units, not 9,900 units
df_clean['amount_log'] = np.log1p(df_clean['amount'])

print(f'  Original range: ${df_clean["amount"].min():.2f} to ${df_clean["amount"].max():,.0f}')
print(f'  Log range:      {df_clean["amount_log"].min():.2f} to {df_clean["amount_log"].max():.2f}')
print(f'  Original std:   {df_clean["amount"].std():.1f}')
print(f'  Log std:        {df_clean["amount_log"].std():.3f}')
print()

# ============================================================
# TREATMENT STRATEGY 4: Binary Indicator Feature
# ============================================================

print('TREATMENT 4: Binary Indicator Feature')
print('-' * 50)

# Create a flag: is this transaction above a high-value threshold?
# This preserves the outlier signal without letting one large value dominate the model
HIGH_VALUE_THRESHOLD = 5000  # Transactions above $5,000
df_clean['is_high_value'] = (df_clean['amount'] > HIGH_VALUE_THRESHOLD).astype(int)

print(f'  High-value threshold: ${HIGH_VALUE_THRESHOLD:,}')
print(f'  High-value transactions: {df_clean["is_high_value"].sum()} ({df_clean["is_high_value"].mean()*100:.1f}%)')
print(f'  Fraud rate among high-value: {df_clean[df_clean["is_high_value"]==1]["is_fraud"].mean()*100:.1f}%')
print(f'  Fraud rate among low-value:  {df_clean[df_clean["is_high_value"]==0]["is_fraud"].mean()*100:.1f}%')

In [ ]:
# Visualize before/after distributions for each treatment
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# --- Plot 1: Raw vs Winsorized ---
# Show the effect of capping on the distribution tail
axes[0, 0].hist(df_clean['amount'].clip(0, 15000), bins=80,
                alpha=0.6, color='#e74c3c', label='Original (capped view to $15K)', density=True)
axes[0, 0].hist(df_clean['amount_winsorized'].clip(0, 15000), bins=80,
                alpha=0.6, color='#3498db', label='Winsorized (p1–p99)', density=True)
axes[0, 0].axvline(x=p99, color='red', linestyle='--', linewidth=1.5,
                   label=f'99th pct = ${p99:,.0f}')
axes[0, 0].set_title('Treatment: Winsorizing (Capping)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Amount ($)')
axes[0, 0].set_ylabel('Density')
axes[0, 0].legend(fontsize=9)

# --- Plot 2: Raw vs Log Transform ---
# The log transform turns a right-skewed distribution into near-normal
axes[0, 1].hist(df_clean['amount'].clip(0, 15000), bins=80,
                alpha=0.7, color='#e74c3c', label='Original Amount ($)', density=True)
ax2_twin = axes[0, 1].twinx()  # Create second y-axis for log-transformed scale
ax2_twin.hist(df_clean['amount_log'], bins=80, alpha=0.5, color='#2ecc71',
              label='log(1 + Amount)', density=True)
axes[0, 1].set_title('Treatment: Log Transform\n(Compresses extreme values)',
                     fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Amount (Red) or log(1+Amount) (Green)')
axes[0, 1].set_ylabel('Density (original)', color='#e74c3c')
ax2_twin.set_ylabel('Density (log)', color='#2ecc71')

# --- Plot 3: Skewness comparison ---
# Skewness: 0 = perfectly symmetric, >0 = right tail, high skewness = many outliers
treatment_names = ['Original', 'Winsorized', 'Log Transform']
skewness_vals = [
    df_clean['amount'].skew(),
    df_clean['amount_winsorized'].skew(),
    df_clean['amount_log'].skew()
]
colors_sk = ['#e74c3c', '#3498db', '#2ecc71']
bars = axes[1, 0].bar(treatment_names, skewness_vals, color=colors_sk,
                       alpha=0.8, edgecolor='black')
for bar, val in zip(bars, skewness_vals):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2,
                   bar.get_height() + 0.1, f'{val:.2f}',
                   ha='center', fontsize=11, fontweight='bold')
axes[1, 0].axhline(y=0, color='black', linewidth=1)
axes[1, 0].axhline(y=1, color='gray', linestyle='--', linewidth=1,
                   label='Moderate skew threshold (1.0)')
axes[1, 0].set_title('Skewness After Each Treatment\n(Closer to 0 = more symmetric)',
                     fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Skewness')
axes[1, 0].legend()

# --- Plot 4: Binary Indicator — fraud rate by value segment ---
# Show that high-value transactions have a different fraud rate
amount_bins = [0, 50, 200, 1000, 5000, 50000, df_clean['amount'].max()]
bin_labels = ['$0-50', '$50-200', '$200-1K', '$1K-5K', '$5K-50K', '>$50K']
df_clean['amount_bin'] = pd.cut(df_clean['amount'], bins=amount_bins,
                                 labels=bin_labels, right=True)

# Calculate fraud rate and count per bin
bin_stats = df_clean.groupby('amount_bin', observed=True).agg(
    fraud_rate=('is_fraud', 'mean'),
    count=('is_fraud', 'count')
).reset_index()

bar_width = 0.6
bar_colors = ['#2ecc71' if r < 0.05 else '#e74c3c' for r in bin_stats['fraud_rate']]
bars = axes[1, 1].bar(bin_stats['amount_bin'], bin_stats['fraud_rate'] * 100,
                       color=bar_colors, alpha=0.8, edgecolor='black', width=bar_width)

# Add count labels
for bar, (_, row) in zip(bars, bin_stats.iterrows()):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                   f'n={row["count"]}', ha='center', fontsize=8)

axes[1, 1].set_title('Fraud Rate by Transaction Amount Bin\n(High-value outliers: signal or noise?)',
                     fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Amount Range')
axes[1, 1].set_ylabel('Fraud Rate (%)')
axes[1, 1].tick_params(axis='x', rotation=30)

plt.suptitle('Outlier Treatment Strategies — Before & After Comparison',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Step: Impact on Model Performance

The ultimate test: does outlier treatment actually improve the machine learning model? Let's compare logistic regression performance under each treatment strategy.

In [ ]:
# ============================================================
# IMPACT ON MODEL: Compare logistic regression under each treatment
# ============================================================

def run_lr_experiment(X, y, label, random_state=42):
    """Train/test split, scale, fit logistic regression, return metrics."""
    # Stratified split: preserve class ratios in both sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state, stratify=y
    )
    # Scale features (fit only on train)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    # Logistic Regression with balanced class weights (since data is imbalanced)
    model = LogisticRegression(class_weight='balanced', max_iter=1000,
                               random_state=random_state)
    model.fit(X_train_s, y_train)

    y_pred = model.predict(X_test_s)
    y_prob = model.predict_proba(X_test_s)[:, 1]

    # Return all key metrics for the fraud class
    return {
        'label':     label,
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall':    recall_score(y_test, y_pred, zero_division=0),
        'f1':        f1_score(y_test, y_pred, zero_division=0),
        'auc_roc':   roc_auc_score(y_test, y_prob),
    }

SHARED_COLS = ['num_transactions_30d', 'merchant_risk', 'hour', 'avg_amount_historical']

# Dataset 1: Raw (includes data errors, untransformed amount)
result_raw = run_lr_experiment(
    df[SHARED_COLS + ['amount']].values,
    df['is_fraud'].values, 'Raw (with errors)'
)

# Dataset 2: After removing data errors, no treatment on amount
result_clean = run_lr_experiment(
    df_clean[SHARED_COLS + ['amount']].values,
    df_clean['is_fraud'].values, 'Errors Removed'
)

# Dataset 3: Error removal + Winsorized amount
result_winsor = run_lr_experiment(
    df_clean[SHARED_COLS + ['amount_winsorized']].values,
    df_clean['is_fraud'].values, 'Winsorized Amount'
)

# Dataset 4: Error removal + Log-transformed amount
result_log = run_lr_experiment(
    df_clean[SHARED_COLS + ['amount_log']].values,
    df_clean['is_fraud'].values, 'Log-Transformed Amount'
)

# Dataset 5: Error removal + Log + Binary high-value indicator
X_combined = df_clean[SHARED_COLS + ['amount_log', 'is_high_value']].values
result_combined = run_lr_experiment(
    X_combined,
    df_clean['is_fraud'].values, 'Log + High-Value Indicator'
)

# Compile results
results = [result_raw, result_clean, result_winsor, result_log, result_combined]
results_df = pd.DataFrame(results).set_index('label')

print('MODEL PERFORMANCE UNDER DIFFERENT OUTLIER TREATMENTS')
print('='*70)
print(results_df.round(4).to_string())
print()
print('Key takeaway: Appropriate treatment of outliers consistently improves model performance.')
print('The log transform is particularly effective for right-skewed features like transaction amounts.')

In [ ]:
# Final visualization: model comparison across treatments
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

labels = results_df.index.tolist()
x = np.arange(len(labels))

# --- Left: Bar chart of recall and AUC-ROC ---
width = 0.35
b1 = axes[0].bar(x - width/2, results_df['recall'], width,
                  label='Recall (fraud caught)', color='#e74c3c', alpha=0.85)
b2 = axes[0].bar(x + width/2, results_df['auc_roc'], width,
                  label='AUC-ROC', color='#3498db', alpha=0.85)

# Add value labels
for bar in b1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)
for bar in b2:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=20, ha='right', fontsize=9)
axes[0].set_title('Recall & AUC-ROC by Treatment Strategy',
                  fontsize=12, fontweight='bold')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1.1)
axes[0].legend()

# --- Right: F1 score comparison (summary metric) ---
bar_colors = ['#95a5a6', '#f39c12', '#e67e22', '#27ae60', '#2980b9']
bars = axes[1].barh(labels, results_df['f1'], color=bar_colors,
                    alpha=0.85, edgecolor='black', linewidth=0.7)

# Highlight the best performer
best_idx = results_df['f1'].argmax()
bars[best_idx].set_edgecolor('gold')
bars[best_idx].set_linewidth(3)

for bar, val in zip(bars, results_df['f1']):
    axes[1].text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=10, fontweight='bold')

axes[1].set_title('F1-Score (Fraud Class) by Treatment\n(Gold border = best)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('F1-Score')
axes[1].set_xlim(0, 1.0)
axes[1].invert_yaxis()  # Highest at top

plt.suptitle('Impact of Outlier Treatment on Fraud Detection Model',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'Best treatment strategy: {results_df["f1"].idxmax()}')
print(f'Best F1 score: {results_df["f1"].max():.4f}')
print(f'Improvement over raw: +{(results_df["f1"].max() - results_df.loc["Raw (with errors)", "f1"]):.4f}')

## Summary: Outlier Detection & Treatment Decision Guide

### Which Detection Method?

| Situation | Use |
|-----------|----|
| Normal distribution, quick check | Z-score |
| Skewed distribution (amounts, incomes) | IQR Fence |
| High-dimensional data, no distribution assumption | Isolation Forest |
| Need to find cluster-relative outliers | LOF |
| Production system needing speed | Isolation Forest |

### Which Treatment?

| Situation | Treatment |
|-----------|----------|
| Clear data errors (negative amounts, impossible values) | Remove |
| Extreme values distorting model, not the signal | Winsorize |
| Right-skewed feature (amounts, durations) | Log transform |
| Outliers ARE the signal (fraud transactions) | Keep + binary indicator |
| Unsure — don't destroy signal | Binary indicator (safe choice) |

### The Golden Rule

> **Always ask: WHY is this value an outlier, and what would be lost by treating it?**
> In fraud detection, the answer is often: the outlier IS the fraud. Treat carefully.

---

## Self-Check Questions

---

### Question 1
Your `amount` feature has a max of $890,000 due to one luxury car purchase. Z-score flags it as an outlier. Should you remove it? What information would you lose?

<details>
<summary>Show Answer</summary>

**No — you should not blindly remove it.** The $890,000 transaction is unusual but potentially real (a luxury car, a down payment, a high-value business transaction). Removing it would:

1. **Lose a legitimate data point** that represents a real customer behavior pattern.
2. **Bias the model** toward low-value transactions — the model never learns to evaluate high-value transactions.
3. **Miss potential fraud signal** — if fraudsters target high-value transactions, removing them destroys detection capability.

**Better approaches:**
- **Investigate first:** Is this amount plausible? Is the merchant category consistent (car dealership vs. grocery store)?
- **Keep and transform:** Apply log transform to reduce the extreme influence while keeping the signal.
- **Create an indicator:** `is_very_high_value = (amount > $100,000)` — preserves signal without numeric distortion.
- **Only remove** if you can verify it's a data error (e.g., the merchant category is a coffee shop with a $890K transaction).

</details>

---

### Question 2
IQR method flags 3% of your training data as outliers. Your test set gets 6% flagged. What might explain this discrepancy?

<details>
<summary>Show Answer</summary>

**Several possibilities:**

1. **Different time periods:** If training data is from January–June and test from July–December, seasonal patterns (holiday shopping, year-end bonuses) might shift transaction distributions.

2. **Distribution shift (concept drift):** The underlying data-generating process changed — new customer segments, new merchant types, economic events.

3. **Small test set:** If the test set is small, 3% vs 6% might be sampling variance rather than a real difference.

4. **IQR calculated on training data only (correct):** The IQR fences are based on training data statistics. If the test set genuinely has more extreme values, they'll be flagged at a higher rate — this is expected and informative.

5. **Fraud pattern change:** New fraud methods might involve different transaction sizes, making outlier patterns more frequent in recent data.

**Action:** Investigate the distribution of the 6% flagged in test data. Are they concentrated in specific features, time periods, or merchant categories? This points to the root cause.

</details>

---

### Question 3
In a fraud detection context, high-value outlier transactions might actually BE the fraudulent ones. What treatment strategy would preserve this signal?

<details>
<summary>Show Answer</summary>

**Best strategies that preserve the fraud signal:**

1. **Binary indicator feature:** `is_high_value = (amount > threshold).astype(int)` — the model gets a direct flag that this is an unusual amount, without the extreme value distorting numeric calculations.

2. **Log transform + keep:** `log_amount = np.log1p(amount)` — compresses the scale dramatically (a $1,000,000 transaction becomes log(1,000,001) ≈ 13.8, close to log(10,000) ≈ 9.2 rather than 100x larger). The outlier is still distinguishable, just with less extreme influence.

3. **Quantile-based features:** `amount_percentile = amount.rank(pct=True)` — converts to a 0–1 rank that preserves ordering without extreme numerical influence.

4. **Keep as-is for tree-based models:** Decision trees and random forests are robust to outliers because they split on thresholds, not distances. A $1M transaction is correctly placed by trees without transformation.

**What NOT to do:** Remove high-value transactions or aggressively cap them — this would make your model blind to high-value fraud.

</details>

---

### Question 4
Isolation Forest has a `contamination` parameter (expected fraction of outliers). What happens if you set it too high? Too low?

<details>
<summary>Show Answer</summary>

**`contamination` controls the threshold used to classify points as outliers after computing anomaly scores.**

**Too high (e.g., contamination=0.3 when true outliers are 2%):**
- The threshold is set to flag 30% of data as outliers
- Many normal transactions get flagged as anomalies
- High false positive rate — investigators waste time on legitimate transactions
- If you're using IF for feature engineering, 30% of your training data gets marked as outliers, potentially removing useful normal-behavior patterns

**Too low (e.g., contamination=0.001 when true outliers are 5%):**
- The threshold is set to flag only 0.1% of data as outliers
- Most actual outliers (including fraud) are NOT flagged
- Model misses the anomalies it should be detecting
- Low recall for the outlier/fraud class

**Best practice:**
- Set contamination close to the known/estimated true proportion of outliers
- When unknown: try multiple values (0.01, 0.05, 0.1) and evaluate on labeled data
- Use the anomaly score (`decision_function`) directly and tune the threshold separately — this decouples the model fitting from the classification threshold

</details>